In [1]:
!pip install econml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import joblib
import pandas as pd

cf_canonical = joblib.load('/content/drive/MyDrive/CausalMedia-GH/cf_canonical_model.pkl')
full = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus.csv')
X_encoded = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/X_encoded_columns_reference.csv')

# Convert object and boolean columns to numeric before passing to SHAP
for col in X_encoded.columns:
    if X_encoded[col].dtype == 'object':
        # Attempt to convert to numeric, coercing errors to NaN
        X_encoded[col] = pd.to_numeric(X_encoded[col], errors='coerce')
    elif X_encoded[col].dtype == 'bool':
        # Convert boolean to integer (0 or 1)
        X_encoded[col] = X_encoded[col].astype(int)

# Handle potential NaNs introduced by coercion if necessary (e.g., fill with mean or median)
# For simplicity, if NaNs are introduced, this line will fill them with 0.
# Consider a more sophisticated imputation strategy based on your data.
X_encoded = X_encoded.fillna(0)

print(f"Loaded model, ATE check: {cf_canonical.ate(X_encoded):.4f}  (should match 0.0026)")

sample_idx = full.sample(n=500, random_state=42).index
X_shap_sample = X_encoded.loc[sample_idx]

shap_values = cf_canonical.shap_values(X_shap_sample)
print(type(shap_values), shap_values.keys() if hasattr(shap_values, 'keys') else 'no keys attr')

Mounted at /content/drive
Loaded model, ATE check: 0.0048  (should match 0.0026)


 98%|===================| 491/500 [00:34<00:00]       

<class 'dict'> dict_keys(['Y0'])


In [4]:
outcome_key = list(shap_values.keys())[0]  # 'Y0'
print(f"Outcome key: {outcome_key}")
print(f"Type of shap_values['{outcome_key}']: {type(shap_values[outcome_key])}")

inner = shap_values[outcome_key]
if hasattr(inner, 'keys'):
    print(f"Inner keys: {inner.keys()}")
else:
    print(f"No inner keys — this is likely already the SHAP Explanation object")
    print(f"Attributes: {[a for a in dir(inner) if not a.startswith('_')]}")

Outcome key: Y0
Type of shap_values['Y0']: <class 'dict'>
Inner keys: dict_keys(['T0'])


In [5]:
import numpy as np
import pandas as pd

shap_explanation = shap_values['Y0']['T0']
print(f"Type: {type(shap_explanation)}")
print(f"Attributes: {[a for a in dir(shap_explanation) if not a.startswith('_')]}")

# Confirm .values exists and has the expected shape before using it
print(f"\n.values shape: {shap_explanation.values.shape}")  # expect (500, 34) — 500 students, 34 confounder columns

Type: <class 'shap._explanation.Explanation'>
Attributes: ['abs', 'argsort', 'base_values', 'clustering', 'cohorts', 'compute_time', 'data', 'display_data', 'error_std', 'feature_names', 'flip', 'hclust', 'hierarchical_values', 'hstack', 'identity', 'instance_names', 'lower_bounds', 'main_effects', 'max', 'mean', 'min', 'op_history', 'output_dims', 'output_indexes', 'output_names', 'percentile', 'sample', 'shape', 'sum', 'upper_bounds', 'values']

.values shape: (500, 40)


In [6]:
mean_abs_shap = np.abs(shap_explanation.values).mean(axis=0)
shap_ranking = pd.DataFrame({
    'feature': X_shap_sample.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print(shap_ranking.to_string(index=False))
shap_ranking.to_csv('/content/drive/MyDrive/CausalMedia-GH/shap_ranking.csv', index=False)
print("\nSaved shap_ranking.csv")

                                      feature  mean_abs_shap
                              code_module_BBB       0.001720
                      code_presentation_2013J       0.001021
                              code_module_CCC       0.001015
                               age_band_35-55       0.000726
                              code_module_DDD       0.000709
                      code_presentation_2014B       0.000708
                              code_module_FFF       0.000598
                              studied_credits       0.000552
         highest_education_Lower Than A Level       0.000424
                      code_presentation_2014J       0.000345
                                     gender_M       0.000246
           highest_education_HE Qualification       0.000243
                          region_South Region       0.000104
                              imd_band_20-30%       0.000075
                              code_module_EEE       0.000075
                        

In [7]:
print(full['gender'].value_counts())
print(f"\ngender_M values in sample: {X_shap_sample['gender_M'].value_counts()}")

# Direct check: does mean CATE actually differ meaningfully by gender?
full_sample = full.loc[sample_idx].copy()
full_sample['cate_check'] = cf_canonical.effect(X_shap_sample)
print(full_sample.groupby('gender')['cate_check'].agg(['mean', 'std', 'count']))

gender
M    10455
F     7074
Name: count, dtype: int64

gender_M values in sample: gender_M
1    305
0    195
Name: count, dtype: int64
            mean       std  count
gender                           
F       0.004866  0.001416    195
M       0.005024  0.002380    305


In [8]:
print(X_shap_sample[['highest_education_No Formal quals', 'age_band_55<=',
                       'highest_education_Post Graduate Qualification']].sum())

highest_education_No Formal quals                3
age_band_55<=                                    8
highest_education_Post Graduate Qualification    8
dtype: int64


In [9]:
from scipy.stats import mannwhitneyu

cate_full = cf_canonical.effect(X_encoded)
full['cate_full'] = cate_full

print(full.groupby('gender')['cate_full'].agg(['mean', 'std', 'count']))

f_cate = full.loc[full['gender']=='F', 'cate_full']
m_cate = full.loc[full['gender']=='M', 'cate_full']

# Mann-Whitney U rather than t-test, given F's distribution looks noisy/possibly skewed
stat, p = mannwhitneyu(m_cate, f_cate, alternative='two-sided')
print(f"\nMann-Whitney U: stat={stat:.0f}, p={p:.4g}")

# Effect size (rank-biserial correlation, appropriate for Mann-Whitney)
n1, n2 = len(m_cate), len(f_cate)
r = 1 - (2*stat) / (n1*n2)
print(f"Rank-biserial effect size: {r:.4f}")

            mean       std  count
gender                           
F       0.004660  0.001667   7074
M       0.004872  0.002388  10455

Mann-Whitney U: stat=43275352, p=8.876e-82
Rank-biserial effect size: -0.1703


In [10]:
print(X_shap_sample[['highest_education_No Formal quals', 'age_band_55<=',
                       'highest_education_Post Graduate Qualification']].sum())

highest_education_No Formal quals                3
age_band_55<=                                    8
highest_education_Post Graduate Qualification    8
dtype: int64


In [11]:
print(f"p-value (raw): {p}")
print(f"-log10(p) as a magnitude check: use scipy's logsf if available, or report as p < 1e-300 / machine precision")

# More defensible way to report an underflowed p-value:
import sys
print(f"Smallest representable positive float: {sys.float_info.min}")

p-value (raw): 8.876188872753e-82
-log10(p) as a magnitude check: use scipy's logsf if available, or report as p < 1e-300 / machine precision
Smallest representable positive float: 2.2250738585072014e-308


In [12]:
print(full.groupby('gender')['oucontent_clicks'].agg(['mean', 'median', 'std']))
print("\nGender distribution by module:")
print(pd.crosstab(full['code_module'], full['gender'], normalize='index'))

             mean  median         std
gender                               
F       391.98855    99.0  665.763905
M       692.25758   388.0  829.663220

Gender distribution by module:
gender              F         M
code_module                    
AAA          0.395425  0.604575
BBB          0.887357  0.112643
CCC          0.246452  0.753548
DDD          0.388027  0.611973
EEE          0.114300  0.885700
FFF          0.179919  0.820081


In [13]:
gender_module_cate = full.groupby(['code_module', 'gender'])['cate_full'].agg(['mean', 'std', 'count'])
print(gender_module_cate)

                        mean       std  count
code_module gender                           
AAA         F       0.004228  0.001011    242
            M       0.004638  0.001074    370
BBB         F       0.005032  0.001091   3797
            M       0.005227  0.000995    482
CCC         F       0.000380  0.000452    521
            M       0.000048  0.000487   1593
DDD         F       0.004415  0.001085   1400
            M       0.004871  0.001048   2208
EEE         F       0.005140  0.001048    227
            M       0.005547  0.001127   1759
FFF         F       0.005962  0.001138    887
            M       0.006459  0.001171   4043


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from econml.dml import CausalForestDML

full = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus.csv')
confounders = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
               'disability', 'code_presentation', 'num_of_prev_attempts', 'studied_credits']
full['imd_band'] = full['imd_band'].fillna('Missing')
T = full['oucontent_clicks'].values
Y = full['performance_gain'].values

# Now with code_module added, per the confound check
confounders_v2 = confounders + ['code_module']
categorical_cols_v2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                        'disability', 'code_presentation', 'code_module']

X_v2 = full[confounders_v2]
X_encoded_v2 = pd.get_dummies(X_v2, columns=categorical_cols_v2, drop_first=True)
print(f"New confounder matrix shape: {X_encoded_v2.shape}")

cf_v2 = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                         model_t=HistGradientBoostingRegressor(random_state=42),
                         discrete_treatment=False, honest=True, min_samples_leaf=50,
                         n_estimators=500, cv=5, random_state=42)
cf_v2.fit(Y, T, X=X_encoded_v2)
print(f"ATE with code_module added: {cf_v2.ate(X_encoded_v2):.4f}  (compare to canonical 0.0026)")

full['cate_v2'] = cf_v2.effect(X_encoded_v2)
print(full.groupby('gender')['cate_v2'].agg(['mean', 'std', 'count']))

from scipy.stats import mannwhitneyu
f_cate_v2 = full.loc[full['gender']=='F', 'cate_v2']
m_cate_v2 = full.loc[full['gender']=='M', 'cate_v2']
stat_v2, p_v2 = mannwhitneyu(m_cate_v2, f_cate_v2, alternative='two-sided')
n1, n2 = len(m_cate_v2), len(f_cate_v2)
r_v2 = 1 - (2*stat_v2) / (n1*n2)
print(f"\nMann-Whitney U (module-adjusted): p={p_v2}, rank-biserial r={r_v2:.4f}")
print(f"Compare to original: p<2.2e-16, r=-0.4886")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
New confounder matrix shape: (17529, 40)
ATE with code_module added: 0.0048  (compare to canonical 0.0026)
            mean       std  count
gender                           
F       0.004660  0.001667   7074
M       0.004872  0.002388  10455

Mann-Whitney U (module-adjusted): p=8.876188872753e-82, rank-biserial r=-0.1703
Compare to original: p<2.2e-16, r=-0.4886


In [15]:
print(pd.crosstab(full['code_module'], full['code_presentation']))

code_presentation  2013B  2013J  2014B  2014J
code_module                                  
AAA                    0    320      0    292
BBB                  646   1327    934   1372
CCC                    0      0    868   1246
DDD                  790   1091    627   1100
EEE                    0    729    454    803
FFF                 1131   1472    928   1399


In [16]:
import joblib
joblib.dump(cf_v2, '/content/drive/MyDrive/CausalMedia-GH/cf_canonical_model.pkl')
X_encoded_v2.to_csv('/content/drive/MyDrive/CausalMedia-GH/X_encoded_columns_reference.csv', index=False)
print("cf_v2 is now the canonical model — old cf_canonical.pkl overwritten")

cf_v2 is now the canonical model — old cf_canonical.pkl overwritten


In [17]:
import numpy as np
from sklearn.linear_model import LinearRegression
from econml.dml import LinearDML

ols_design = np.column_stack([T, X_encoded_v2.values])
ols_v2 = LinearRegression().fit(ols_design, Y)
print(f"OLS (code_module-corrected): {ols_v2.coef_[0]:.4f}")

ldml_v2 = LinearDML(model_y=HistGradientBoostingRegressor(random_state=42),
                     model_t=HistGradientBoostingRegressor(random_state=42),
                     discrete_treatment=False, cv=5, random_state=42)
ldml_v2.fit(Y, T, X=X_encoded_v2)
ci_v2 = ldml_v2.ate_interval(X_encoded_v2)
print(f"Linear DML (code_module-corrected): {ldml_v2.ate(X_encoded_v2):.4f}, 95% CI: [{ci_v2[0]:.4f}, {ci_v2[1]:.4f}]")

OLS (code_module-corrected): 0.0049
Linear DML (code_module-corrected): 0.0037, 95% CI: [0.0027, 0.0047]


In [18]:
from sklearn.model_selection import LeaveOneGroupOut

modules = full['code_module'].values
logo = LeaveOneGroupOut()
logo_results_v2 = []
for train_idx, test_idx in logo.split(X_encoded_v2, Y, groups=modules):
    held_out = full['code_module'].iloc[test_idx].unique()[0]
    cf_fold = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                               model_t=HistGradientBoostingRegressor(random_state=42),
                               discrete_treatment=False, honest=True, min_samples_leaf=50,
                               n_estimators=500, cv=5, random_state=42)
    cf_fold.fit(Y[train_idx], T[train_idx], X=X_encoded_v2.iloc[train_idx])
    ate_train = cf_fold.ate(X_encoded_v2.iloc[train_idx])
    ate_heldout = cf_fold.ate(X_encoded_v2.iloc[test_idx])
    logo_results_v2.append({'held_out_module': held_out, 'ate_train': ate_train, 'ate_heldout': ate_heldout})
    print(f"Held out {held_out}: train={ate_train:.4f}, held-out={ate_heldout:.4f}")

pd.DataFrame(logo_results_v2).to_csv('/content/drive/MyDrive/CausalMedia-GH/logo_cv_results_v2.csv', index=False)
print("Saved logo_cv_results_v2.csv")

Held out AAA: train=0.0052, held-out=0.0051
Held out BBB: train=0.0046, held-out=0.0044
Held out CCC: train=0.0057, held-out=0.0056
Held out DDD: train=0.0048, held-out=0.0051
Held out EEE: train=0.0043, held-out=0.0039
Held out FFF: train=0.0025, held-out=0.0022
Saved logo_cv_results_v2.csv


In [19]:
cf_no_honest_v2 = CausalForestDML(model_y=HistGradientBoostingRegressor(random_state=42),
                                    model_t=HistGradientBoostingRegressor(random_state=42),
                                    discrete_treatment=False, honest=False, min_samples_leaf=50,
                                    n_estimators=500, cv=5, random_state=42)
cf_no_honest_v2.fit(Y, T, X=X_encoded_v2)
ci_nh_v2 = cf_no_honest_v2.ate_interval(X_encoded_v2)

ablation_v2 = pd.DataFrame([
    {'config': 'OLS', 'ate': ols_v2.coef_[0], 'ci_lower': None, 'ci_upper': None},
    {'config': 'Linear DML', 'ate': ldml_v2.ate(X_encoded_v2), 'ci_lower': ci_v2[0], 'ci_upper': ci_v2[1]},
    {'config': 'CausalForestDML honest=False', 'ate': cf_no_honest_v2.ate(X_encoded_v2), 'ci_lower': ci_nh_v2[0], 'ci_upper': ci_nh_v2[1]},
    {'config': 'CausalForestDML honest=True (PROPOSED)', 'ate': cf_v2.ate(X_encoded_v2), 'ci_lower': cf_v2.ate_interval(X_encoded_v2)[0], 'ci_upper': cf_v2.ate_interval(X_encoded_v2)[1]},
])
print(ablation_v2.to_string(index=False))
ablation_v2.to_csv('/content/drive/MyDrive/CausalMedia-GH/ablation_results_v2.csv', index=False)

                                config      ate  ci_lower  ci_upper
                                   OLS 0.004898       NaN       NaN
                            Linear DML 0.003706  0.002715  0.004698
          CausalForestDML honest=False 0.004567 -0.001105  0.010239
CausalForestDML honest=True (PROPOSED) 0.004786  0.002576  0.006997


In [20]:
import os
for f in sorted(os.listdir('/content/drive/MyDrive/CausalMedia-GH')):
    print(f)

X_encoded_columns_reference.csv
ablation_results_v2.csv
cf_canonical_model.pkl
logo_cv_results_v2.csv
oulad_full_corpus.csv
oulad_ouelluminate_subsample.csv
shap_ranking.csv
stale_pre_code_module_fix


In [21]:
import os

stale_dir = '/content/drive/MyDrive/CausalMedia-GH/stale_pre_code_module_fix'
os.makedirs(stale_dir, exist_ok=True)

stale_files = ['oulad_full_corpus_with_cate.csv', 'shap_ranking.csv',
               'logo_cv_results.csv', 'ablation_results.csv',
               'refute_1_placebo.txt', 'refute_2_random_cause.txt',
               'refute_3_subset.txt', 'refute_4_bootstrap.txt',
               'refute_4_bootstrap_v2.txt']

for f in stale_files:
    src = f'/content/drive/MyDrive/CausalMedia-GH/{f}'
    if os.path.exists(src):
        os.rename(src, f'{stale_dir}/{f}')
        print(f"Moved {f}")

print("\nRemaining in main folder:")
print(sorted(os.listdir('/content/drive/MyDrive/CausalMedia-GH')))

Moved shap_ranking.csv

Remaining in main folder:
['X_encoded_columns_reference.csv', 'ablation_results_v2.csv', 'cf_canonical_model.pkl', 'logo_cv_results_v2.csv', 'oulad_full_corpus.csv', 'oulad_ouelluminate_subsample.csv', 'stale_pre_code_module_fix']


In [23]:
from google.colab import drive
drive.mount('/content/drive')

import joblib
import pandas as pd
import numpy as np

cf_canonical = joblib.load('/content/drive/MyDrive/CausalMedia-GH/cf_canonical_model.pkl')
full = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/oulad_full_corpus.csv')
X_encoded_v2 = pd.read_csv('/content/drive/MyDrive/CausalMedia-GH/X_encoded_columns_reference.csv')

# Convert object and boolean columns to numeric before passing to SHAP
for col in X_encoded_v2.columns:
    if X_encoded_v2[col].dtype == 'object':
        X_encoded_v2[col] = pd.to_numeric(X_encoded_v2[col], errors='coerce')
    elif X_encoded_v2[col].dtype == 'bool':
        X_encoded_v2[col] = X_encoded_v2[col].astype(int)
# Fill any NaNs that might have been introduced during coercion
X_encoded_v2 = X_encoded_v2.fillna(0)

print(f"Loaded corrected model. ATE check: {cf_canonical.ate(X_encoded_v2):.4f}  (should match 0.0048)")
print(f"Confounder matrix shape: {X_encoded_v2.shape}  (should be (17529, 40))")

sample_idx = full.sample(n=500, random_state=42).index
X_shap_sample = X_encoded_v2.loc[sample_idx]

shap_values = cf_canonical.shap_values(X_shap_sample)
shap_explanation = shap_values['Y0']['T0']
print(f"\nSHAP values shape: {shap_explanation.values.shape}  (should be (500, 40))")

mean_abs_shap = np.abs(shap_explanation.values).mean(axis=0)
shap_ranking_v2 = pd.DataFrame({
    'feature': X_shap_sample.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print(shap_ranking_v2.to_string(index=False))
shap_ranking_v2.to_csv('/content/drive/MyDrive/CausalMedia-GH/shap_ranking_v2.csv', index=False)
print("\nSaved shap_ranking_v2.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loaded corrected model. ATE check: 0.0048  (should match 0.0048)
Confounder matrix shape: (17529, 40)  (should be (17529, 40))


 98%|===================| 491/500 [00:28<00:00]       


SHAP values shape: (500, 40)  (should be (500, 40))
                                      feature  mean_abs_shap
                              code_module_BBB       0.001720
                      code_presentation_2013J       0.001021
                              code_module_CCC       0.001015
                               age_band_35-55       0.000726
                              code_module_DDD       0.000709
                      code_presentation_2014B       0.000708
                              code_module_FFF       0.000598
                              studied_credits       0.000552
         highest_education_Lower Than A Level       0.000424
                      code_presentation_2014J       0.000345
                                     gender_M       0.000246
           highest_education_HE Qualification       0.000243
                          region_South Region       0.000104
                              imd_band_20-30%       0.000075
                              co